# HW5. Локальная разработка

Все реализации находятся в prepare.py, rewards.py и train.py. Notebook импортирует их и показывает промежуточные результаты. После изменения модулей перезапусти kernel и выполни ячейки сверху вниз, чтобы проверить актуальный код.

Здесь используем 50 настоящих диалогов ToolACE, настоящий токенизатор и подставленные ответы. Веса модели и GPU не нужны. Это проверка подготовки и награды; короткое настоящее RL-обучение находится в gpu_smoke.ipynb. Baseline и дальнейшие эксперименты — в experiments.ipynb.

In [1]:
import json
from pathlib import Path
from pprint import pprint

from prepare import (
    parse_dialogues, split_dialogues, make_rl_examples,
    render_prompt, save_examples, load_examples,
)
from rewards import parse_qwen_calls, reward, score_response, format_reference
from train import load_tokenizer, build_training_rows

## 1. Настоящие исходные данные

Если data/raw/data.json отсутствует, сначала выполни в терминале python download_data.py. Он скачивает закреплённую версию ToolACE. В notebook читаем локальный файл и берём первые 50 записей.

In [2]:
with Path('data/raw/data.json').open(encoding='utf-8') as file:
    data = json.load(file)
sample = data[:50]
print(f'Исходных диалогов: {len(data)}, локальный сэмпл: {len(sample)}')
for message in sample[0]['conversations'][:3]:
    print(f"\n--- {message['from']} ---")
    print(message['value'])

Исходных диалогов: 11300, локальный сэмпл: 50

--- user ---
I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US?

--- assistant ---
[Market Trends API(trend_type="MARKET_INDEXES", country="us")]

--- tool ---
[{"name": "Market Trends API", "results": {"trends": [{"name": "S&P 500", "description": "Standard & Poor's 500 Index is a market-capitalization-weighted index of the 500 largest U.S. publicly traded companies.", "data": {"current_value": "4172.80", "percentage_change": "+0.68%"}}, {"name": "DOW J", "description": "Dow Jones Industrial Average is a price-weighted average of 30 blue-chip stocks that are generally the leaders in their industry.", "data": {"current_value": "34479.60", "percentage_change": "+0.47%"}}, {"name": "NASDAQ", "description": "The NASDAQ Composite is a broad-based capitalization-weighted index of stocks in all three NASDAQ tiers: Global Select, Global Market and Capital Mark

## 2. Разбор источника

Инструменты извлекаются из system. Вызовы assistant превращаются в структурированные tool_calls. Диалоги с ошибками исключаются целиком и перечисляются ниже.

In [3]:
parsed_dialogues, parse_errors = parse_dialogues(sample)
print(f'Разобрано: {len(parsed_dialogues)}, исключено: {len(parse_errors)}')
pprint(parse_errors)
print('Первый разобранный вызов:')
pprint(parsed_dialogues[0]['messages'][1])

Разобрано: 48, исключено: 2
[{'dialogue_id': 27, 'reason': 'Пустая реплика assistant: 1'},
 {'dialogue_id': 30, 'reason': 'Пустая реплика assistant: 1'}]
Первый разобранный вызов:
{'content': '',
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': {'country': 'us',
                                            'trend_type': 'MARKET_INDEXES'},
                              'name': 'Market Trends API'},
                 'type': 'function'}]}


## 3. Разделение и RL-примеры

Сначала разделяем целые диалоги, затем нарезаем каждый перед репликами assistant. Один пример — тройка (prompt, answer, tools). В prompt находятся предыдущие сообщения; answer содержит отдельную целевую реплику.

In [4]:
train_dialogues, validation_dialogues = split_dialogues(parsed_dialogues)
train_examples = make_rl_examples(train_dialogues)
validation_examples = make_rl_examples(validation_dialogues)
train_ids = {dialogue['dialogue_id'] for dialogue in train_dialogues}
validation_ids = {dialogue['dialogue_id'] for dialogue in validation_dialogues}
assert train_ids.isdisjoint(validation_ids)
print(f'Train: {len(train_examples)}, validation: {len(validation_examples)}')
prompt, answer, tools = train_examples[0]
print('История:')
pprint(prompt)
print('Отдельный эталон:')
pprint(answer)

Train: 208, validation: 27
История:
[{'content': 'Could you help me find some trending hashtags on TikTok right '
             'now?',
  'role': 'user'}]
Отдельный эталон:
{'content': "I don't have the capability to directly fetch real-time trends or "
            'the current trending hashtags on TikTok. However, if you have a '
            'specific hashtag in mind, I can retrieve detailed information '
            'about it. Could you please specify the hashtag you are interested '
            'in?',
 'role': 'assistant'}


## 4. Вход Qwen

Загружаем только токенизатор закреплённой ревизии. Сравни напечатанный конец prompt с историей выше: там должен находиться запрос и начало будущего ответа, без текущего эталона.

In [5]:
tokenizer = load_tokenizer()
text = render_prompt(prompt, tools, tokenizer)
print(text[-2000:])
print('Токенов:', len(tokenizer(text, add_special_tokens=False)['input_ids']))

customizable messages and authentication.", "parameters": {"type": "object", "properties": {"urls": {"description": "List of URLs of the LinkedIn users to send connection requests to.", "type": "string", "format": "comma-separated"}, "message": {"description": "Customizable message to send with the connection request.", "type": "string"}, "cookie": {"description": "Value of the 'li_at' cookie for authentication.", "type": "string"}, "email": {"description": "Email address to receive the results of connection requests' status.", "type": "string"}, "key": {"description": "API key for authentication.", "type": "string"}}, "required": ["urls", "message", "cookie", "email", "key"]}}}
{"type": "function", "function": {"name": "Get Instagram Media Information", "description": "Retrieves media information for an Instagram post, reel, or TV by providing a URL.", "parameters": {"type": "object", "properties": {"link": {"description": "The URL of the Instagram post, reel, or TV.", "type": "string

## 5. Разбор подставленного ответа

Берём настоящий эталон с вызовом и записываем его в нативной разметке Qwen. Этот готовый текст заменяет генерацию только для локальной проверки. Формирование ответа и его разбор выполняются разными функциями.

In [6]:
for prompt, answer, tools in train_examples:
    if answer.get('tool_calls'):
        break
response = format_reference(answer)
print(response)
print('Разобранные вызовы:')
pprint(parse_qwen_calls(response, tools))
print('Эталонные вызовы:')
pprint(answer['tool_calls'])

<tool_call>
<function=Hashtag Info API>
<parameter=hashtag>
funnychallenge
</parameter>
</function>
</tool_call>
Разобранные вызовы:
[{'arguments': {'hashtag': 'funnychallenge'}, 'name': 'Hashtag Info API'}]
Эталонные вызовы:
[{'function': {'arguments': {'hashtag': 'funnychallenge'},
               'name': 'Hashtag Info API'},
  'type': 'function'}]


## 6. Награды

exact — полное совпадение набора вызовов и аргументов с учётом повторов; functions — F1 по именам функций с учётом повторов; format — ответ успешно разобран, имена и обязательные параметры известны. Все компоненты лежат в диапазоне 0–1. Для корректного непустого ответа без вызовов при таком же эталоне все компоненты равны 1; смысл текста эта награда не оценивает.

reward сохраняет нашу исходную бинарную награду exact. Серверный smoke использует явно заданную смесь компонентов, чтобы видеть более подробный сигнал. format не является полной проверкой JSON Schema.

In [7]:
cases = [
    ('Корректный вызов', response, 1),
    ('Пропущенный вызов', 'Ответ без вызова инструмента.', 0),
    ('Сломанная разметка', '<tool_call>', 0),
]
for name, candidate, expected in cases:
    actual = reward(candidate, answer, tools)
    print(name, 'ожидаем:', expected, 'получено:', actual)
    pprint(score_response(candidate, answer, tools))
    assert actual == expected

Корректный вызов ожидаем: 1 получено: 1
{'exact': 1.0, 'format': 1.0, 'functions': 1.0}
Пропущенный вызов ожидаем: 0 получено: 0
{'exact': 0.0, 'format': 1.0, 'functions': 0.0}
Сломанная разметка ожидаем: 0 получено: 0
{'exact': 0.0, 'format': 0.0, 'functions': 0.0}


## 7. Пакет и сохранение

Проверяем все подготовленные примеры на подставленных корректных ответах. Затем сохраняем тройки в JSONL и читаем обратно. Это проверяет передачу эталонов, инструментов и сохранность данных; качество реальной генерации проверит серверный notebook.

In [8]:
scores = []
for prompt, answer, tools in train_examples + validation_examples:
    scores.append(reward(format_reference(answer), answer, tools))
print(f'Корректно обработано: {sum(scores)} / {len(scores)}')
assert all(score == 1 for score in scores)
output_dir = Path('data/local_prepared')
output_dir.mkdir(parents=True, exist_ok=True)
save_examples(train_examples, output_dir / 'train.jsonl')
save_examples(validation_examples, output_dir / 'validation.jsonl')
assert load_examples(output_dir / 'train.jsonl') == train_examples
print('JSONL прочитан без изменения данных.')

Корректно обработано: 235 / 235
JSONL прочитан без изменения данных.


## 8. Данные для TRL

Сервер использует те же функции подготовки и награды. Prompt заранее отрендерен со своими инструментами. Эталон и схемы хранятся в отдельных JSON-строках: Dataset не объединит разные наборы аргументов в один словарь с добавленными null. Эти колонки используются наградой, а не попадают во вход модели.

In [9]:
rows, skipped = build_training_rows(train_examples, tokenizer, max_prompt_tokens=2048)
print('Пригодных примеров:', len(rows), 'исключено по длине:', skipped)
print('Колонки:', list(rows[0]))
print(rows[0]['prompt'][-500:])
print('Отдельный эталон:')
pprint(json.loads(rows[0]['answer_json']))

Пригодных примеров: 194 исключено по длине: 14
Колонки: ['prompt', 'answer_json', 'tools_json']
 <tool_call></tool_call> XML tags
- Required parameters MUST be specified
- You may provide optional reasoning for your function call in natural language BEFORE the function call, but NOT after
- If there is no function call available, answer the question like normal with your current knowledge and do not tell the user about function calls
</IMPORTANT><|im_end|>
<|im_start|>user
Could you help me find some trending hashtags on TikTok right now?<|im_end|>
<|im_start|>assistant
<think>

</think>


Отдельный эталон:
{'content': "I don't have the capability to directly fetch real-time trends or "
            'the current trending hashtags on TikTok. However, if you have a '
            'specific hashtag in mind, I can retrieve detailed information '
            'about it. Could you please specify the hashtag you are interested '
            'in?',
 'role': 'assistant'}
